# Assignment 1: EDA and Baseline
GROUP50_CENG313, LLM Classification Finetuning

In [ ]:
# Colab: clone the repo so src/ can be imported
# !git clone https://github.com/akdagdev/GROUP50_CENG313.git
# %cd GROUP50_CENG313
from src.data import load_train, load_test
from src.utils import set_seed
set_seed(42)

## 1. Data overview


In [ ]:
import pandas as pd

df = pd.read_csv("/content/train.csv")
print(df.shape)

## 2. Data quality (missing values, duplicates, inconsistencies)

## 3. Exploratory analysis and visualizations

In [ ]:
import json
import numpy as np
import pandas as pd

def kelime_sayisi(cevap):
    turlar = json.loads(cevap)
    return sum(len(tur.split()) for tur in turlar if isinstance(tur, str))

a_uzunluk = df["response_a"].map(kelime_sayisi)
b_uzunluk = df["response_b"].map(kelime_sayisi)

grup = np.select(
    [a_uzunluk > b_uzunluk, b_uzunluk > a_uzunluk],
    ["A daha uzun", "B daha uzun"],
    default="Eşit uzunluk",
)

sonuc = df[["winner_model_a", "winner_model_b", "winner_tie"]].idxmax(axis=1)
sonuc = sonuc.map({
    "winner_model_a": "A kazandı",
    "winner_model_b": "B kazandı",
    "winner_tie": "Beraberlik",
})

oranlar = pd.crosstab(grup, sonuc, normalize="index") * 100
display(oranlar.round(1))

## 4. Baseline models


In [ ]:
import numpy as np
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

winner_cols = ["winner_model_a", "winner_model_b", "winner_tie"]
y = df[winner_cols].to_numpy().argmax(axis=1)

train_idx, val_idx = train_test_split(
    np.arange(len(df)), test_size=0.20, random_state=42, stratify=y
)

class_probs = np.bincount(y[train_idx], minlength=3) / len(train_idx)
val_probs = np.tile(class_probs, (len(val_idx), 1))

score = log_loss(y[val_idx], val_probs, labels=[0, 1, 2])
print(f"Validation log loss = {score:.4f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X_uzunluk = pd.DataFrame({
    "a_kelime": a_uzunluk,
    "b_kelime": b_uzunluk,
    "kelime_farki": a_uzunluk - b_uzunluk,
})

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_uzunluk.iloc[train_idx], y[train_idx])

tahmin = model.predict_proba(X_uzunluk.iloc[val_idx])
uzunluk_skoru = log_loss(y[val_idx], tahmin, labels=[0, 1, 2])

print(f"Sınıf oranları baseline: {score:.4f}")
print(f"Uzunluk + lojistik regresyon: {uzunluk_skoru:.4f}")

## 5. Results and conclusions

Sınıf oranları baseline: 1.0972
Uzunluk + lojistik regresyon: 1.0709